In [1]:
# Packages for synthetic data generation and pre-processing.  
from sklearn.datasets import make_blobs, make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import pandas as pd

# Qiskit packages
from qiskit.circuit.library import ZZFeatureMap,RealAmplitudes
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.optimizers import SPSA
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Packages for data visualization
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

FILE_NAME = 'Heart_Disease_and_Hospitals.csv'
df = pd.read_csv(FILE_NAME)
print(f"SUCCESS: Data loaded from {FILE_NAME}")

unique_sorted_states = sorted(df['state'].unique())

# Create a mapping dictionary: {'Illinois': 0, 'Iowa': 1, ...}
state_to_num = {state: i for i, state in enumerate(unique_sorted_states)}
print("State Encoding Map:")
print(state_to_num)

# Apply the mapping to the 'state' column
df['state_encoded'] = df['state'].map(state_to_num)


# --- 3. Encode 'gender' (Female=1, Male=0) ---

# Define the binary mapping
gender_map = {'Female': 1, 'Male': 0}

# Apply the mapping to the 'gender' column
df['gender_encoded'] = df['gender'].map(gender_map)

# Get unique treatments and sort them alphabetically
unique_sorted_treatments = sorted(df['treatment'].unique())

# Create a mapping dictionary for treatment
treatment_to_num = {treatment: i for i, treatment in enumerate(unique_sorted_treatments)}
print("\nTreatment Encoding Map:")
print(treatment_to_num)

# Apply the mapping to the 'treatment' column
df['treatment_encoded'] = df['treatment'].map(treatment_to_num)

FEATURE_COLUMNS = ['age', 'state_encoded', 'blood_pressure', 'cholesterol', 'bmi', 'glucose_level', 'gender_encoded', 'treatment_encoded']
N_FEATURES = len(FEATURE_COLUMNS)

# X contains all selected features:
X = df[FEATURE_COLUMNS].values

# y contains the target label (HeartDisease):
y = df['heart_disease'].values

# --- 3. Print Results ---
print("\n--- Data Head ---")
print(df.head()) # Shows the first 5 rows of the loaded DataFrame

print("\n--- Array Dimensions ---")
print(f"Features (X) shape: {X.shape}")
print(f"Labels (y) shape: {y.shape}")
print("\nData is now loaded from the external CSV and is ready for QML.")

TEST_SIZE = 0.2  # 30% of data for testing
RANDOM_SEED = 24 # Ensures reproducibility

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_SEED,
    # stratify=y  # Uncomment if you want proportional split of classes
)

print("\n--- Data Splitting Results ---")
print("Data successfully split into training and testing sets.")
print(f"Total Samples: {X.shape[0]}")
print(f"Training Samples (X_train): {X_train.shape[0]}")
print(f"Testing Samples (X_test): {X_test.shape[0]}")

train_data = X_train
test_data = X_test
train_labels = y_train
test_labels = y_test

feature_map = ZZFeatureMap(feature_dimension=N_FEATURES, reps=1)
feature_map.barrier()
var_form = RealAmplitudes(num_qubits=N_FEATURES, reps=4)
circuit = feature_map.compose(var_form).decompose()
circuit.measure_all()
circuit.decompose().draw(output="mpl", fold=-1, style="iqp")

def circuit_instance(data, variational):
    """
    Assigns parameter values to the quantum circuit.
    """
    parameters = {}
    for i, p in enumerate(feature_map.parameters):
        parameters[p] = data[i]
    for i, p in enumerate(var_form.ordered_parameters):
        parameters[p] = variational[i]
    return circuit.assign_parameters(parameters)

def labels_probability(results):
    """
    Converts a dictionary of bitstrings and their counts, to parities and their counts
    """
    shots = sum(results.values())
    probabilities = {0: 0, 1: 0}
    for bitstring, counts in results.items():
        label = (sum(int(k) for k in list(bitstring))+1) % 2
        probabilities[label] += counts / shots
    return probabilities

def classification_probability_simulation(data, variational):
    """
    Classify data points using given parameters.
    """
    circuits = [circuit_instance(d, variational) for d in data]
    sampler = StatevectorSampler() # Estimator() matrix_state_product
    results = sampler.run(circuits).result()
    # print(results[1].data.meas.get_counts())
    classification = [labels_probability(results[i].data.meas.get_counts()) for i, c in enumerate(circuits)]
    return classification

def cross_entropy_loss(classification, expected):
    """
    Calculate accuracy of predictions using cross entropy loss.
    """
    p = classification.get(expected)  
    return -np.log(p + 1e-10)

def cost_function(data, labels, variational):
    """
    Evaluates performance of our circuit with variational parameters on data.
    """
    classifications = classification_probability_simulation(data, variational)
    cost = 0
    for i, classification in enumerate(classifications):
        cost += cross_entropy_loss(classification, labels[i])
    cost /= len(data)
    return cost

def objective_function(variational):
    """Cost function of circuit parameters on training data.
    The optimizer will attempt to minimize this."""
    return cost_function(train_data, train_labels, variational)

class OptimizerLog:
    """Log to store optimizer's intermediate results"""
    def __init__(self):
        self.evaluations = []
        self.parameters = []
        self.costs = []
    def update(self, evaluation, parameter, cost, _stepsize, _accept):
        """Save intermediate results. Optimizer passes five values
        but we ignore the last two."""
        self.evaluations.append(evaluation)
        self.parameters.append(parameter)
        self.costs.append(cost)
        print(f"Iteration {evaluation + 1}: Cost = {cost:.6f}")
             
# Set up the optimization
log = OptimizerLog()
optimizer = SPSA(maxiter=100, callback=log.update)

initial_point = np.zeros((var_form.num_parameters))

# Run the optimization
result = optimizer.minimize(objective_function, initial_point)

opt_var = result.x
opt_value = result.fun

fig = plt.figure()
plt.plot(log.evaluations, log.costs)
plt.xlabel('Steps')
plt.ylabel('Cost')
plt.grid(True)
plt.rc('axes', linewidth=2)  # Set the axes linewidth globally
ax = plt.gca()  # Get current axes
ax.spines['top'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
plt.show()





SUCCESS: Data loaded from Heart_Disease_and_Hospitals.csv
State Encoding Map:
{'Alabama': 0, 'Alaska': 1, 'Arizona': 2, 'Arkansas': 3, 'California': 4, 'Colorado': 5, 'Connecticut': 6, 'Delaware': 7, 'Florida': 8, 'Georgia': 9, 'Hawaii': 10, 'Idaho': 11, 'Illinois': 12, 'Indiana': 13, 'Iowa': 14, 'Kansas': 15, 'Kentucky': 16, 'Louisiana': 17, 'Maine': 18, 'Maryland': 19, 'Massachusetts': 20, 'Michigan': 21, 'Minnesota': 22, 'Mississippi': 23, 'Missouri': 24, 'Montana': 25, 'Nebraska': 26, 'Nevada': 27, 'New Hampshire': 28, 'New Jersey': 29, 'New Mexico': 30, 'New York': 31, 'North Carolina': 32, 'North Dakota': 33, 'Ohio': 34, 'Oklahoma': 35, 'Oregon': 36, 'Pennsylvania': 37, 'Rhode Island': 38, 'South Carolina': 39, 'South Dakota': 40, 'Tennessee': 41, 'Texas': 42, 'Utah': 43, 'Vermont': 44, 'Virginia': 45, 'Washington': 46, 'West Virginia': 47, 'Wisconsin': 48, 'Wyoming': 49}

Treatment Encoding Map:
{'Counseling': 0, 'Medication': 1, 'Physiotherapy': 2, 'Rehabilitation': 3, 'Surgery

KeyboardInterrupt: 

In [ ]:
def create_conf_mat(y_true, y_pred, title):
    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Normalize the confusion matrix by row (i.e., by the number of samples in each class)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

    # Create a custom colormap
    colors = [(1, 1, 1), (0/255, 255/255, 0/255)] 
    n_bins = 100  # Number of bins in the colormap
    cmap_name = 'custom_blue'
    custom_cmap = LinearSegmentedColormap.from_list(cmap_name, colors, N=n_bins)

    # Display confusion matrix
    fig, ax = plt.subplots()
    cax = ax.matshow(cm_normalized, cmap=custom_cmap)

    # Add color bar
    fig.colorbar(cax)

    # Set predefined RGB color for the text
    for i in range(cm_normalized.shape[0]):
        for j in range(cm_normalized.shape[1]):
            color = 'white' if i == j else 'black'
            ax.text(j, i, f'{cm_normalized[i, j]:.1f}%', va='center', ha='center', color=color)

    # Set predefined background color for the diagonal elements (e.g., RGB (255, 255, 0))
    diagonal_color = (98/255, 35/255, 223/255)  # Normalize to 0-1 range
    for i in range(cm_normalized.shape[0]):
        ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=True, color=diagonal_color, alpha=0.3))

    # Set predefined background color for the plot (e.g., RGB (240, 240, 240))
    background_color = (1, 1, 1)  # Normalize to 0-1 range
    fig.patch.set_facecolor(background_color)
    ax.set_facecolor(background_color)

    plt.title(title)
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.show()

def test_classifier(data, labels, variational):
    """
    Gets classifier'spredictions and calculates accuracy 
    """
    probability = classification_probability_simulation(data, variational)
    predictions = [0 if p[0] >= p[1] else 1 for p in probability]
    accuracy = 0
    for i, prediction in enumerate(predictions):
        if prediction == labels[i]:
            accuracy += 1
    accuracy /= len(labels)
    print(f"Simulated Test Accuracy: {accuracy * 100:.2f}%")
    create_conf_mat(labels, predictions, "Simulated Evaluation")
    return accuracy, predictions

accuracy_sim, predictions_sim = test_classifier(test_data, test_labels, opt_var)
